In [1]:
import psycopg2
import os
from dotenv import load_dotenv
load_dotenv()

conn =  psycopg2.connect(os.getenv("DATABASE_URL"))

In [2]:
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS songs (
    id SERIAL PRIMARY KEY,
    title TEXT NOT NULL,
    artist TEXT,
    duration FLOAT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS fingerprints (
    hash BIGINT NOT NULL,
    song_id INT NOT NULL,
    time_offset INT NOT NULL,
    anchor_freq INT NOT NULL,
    FOREIGN KEY (song_id) REFERENCES songs(id) ON DELETE CASCADE
);
""")

cur.execute("""
CREATE INDEX IF NOT EXISTS idx_fingerprint_hash
ON fingerprints(hash);
""")

cur.execute("""
CREATE INDEX IF NOT EXISTS idx_fingerprint_hash_anchor
ON fingerprints(hash, anchor_freq);
""")

conn.commit()
cur.close()
conn.close()

In [ ]:
from app.utils import generate_hashes_for_audio
hashes = generate_hashes_for_audio("data/misirlou.mp3")

In [34]:
hashes

array([[8478725,       9,      66],
       [8671237,       9,      66],
       [8826885,       9,      66],
       ...,
       [7266333,    5484,     373],
       [7200797,    5484,     373],
       [7475229,    5484,     373]], shape=(56806, 3))

In [28]:
conn =  psycopg2.connect(os.getenv("DATABASE_URL"))
cur = conn.cursor()

cur.execute("""
    INSERT INTO songs (title, artist, duration)
    VALUES (%s, %s, %s)
    RETURNING id;
""", ("Kaun Tujhe", "Kishore Kumar", 195))

song_id = cur.fetchone()[0]

conn.commit()
cur.close()
conn.close()

In [29]:
from psycopg2.extras import execute_values

conn =  psycopg2.connect(os.getenv("DATABASE_URL"))
cur = conn.cursor()

data = [(int(h), int(song_id), int(offset), int(anchor_freq)) for h, offset, anchor_freq in hashes]

query = """
    INSERT INTO fingerprints (hash, song_id, time_offset, anchor_freq)
    VALUES %s
"""

execute_values(cur, query, data, page_size=10000)

conn.commit()
cur.close()
conn.close()

In [30]:
conn = psycopg2.connect(os.getenv("DATABASE_URL"))
cur = conn.cursor()

query_hashes = hashes[:, 0].astype(int).tolist()

if query_hashes:
    cur.execute(
        """
        SELECT hash, song_id, time_offset, anchor_freq
        FROM fingerprints
        WHERE hash = ANY(%s)
        """,
        (query_hashes,),
    )
    results = cur.fetchall()
else:
    results = []

cur.close()
conn.close()

results

[(4399111, 4, 4787, 978),
 (4440079, 4, 4842, 975),
 (4452366, 4, 475, 961),
 (4472837, 4, 4789, 960),
 (4501515, 4, 478, 949),
 (4517893, 4, 4789, 949),
 (4517894, 4, 6066, 955),
 (4542479, 4, 4842, 975),
 (4562952, 4, 278, 950),
 (4595723, 4, 384, 947),
 (4603913, 1, 2346, 936),
 (4603913, 1, 8724, 932),
 (4603913, 4, 264, 948),
 (4612102, 1, 3874, 930),
 (4612102, 4, 134, 952),
 (4612113, 4, 732, 936),
 (4624389, 1, 3575, 958),
 (4624389, 1, 4588, 924),
 (4624389, 1, 5177, 923),
 (4624389, 4, 4789, 923),
 (4628490, 4, 475, 961),
 (4632581, 1, 3603, 922),
 (4632581, 4, 4787, 978),
 (4632585, 1, 7997, 923),
 (4632585, 4, 4887, 944),
 (4636687, 4, 278, 950),
 (4640779, 4, 4814, 976),
 (4661254, 1, 6957, 963),
 (4661254, 4, 6066, 955),
 (4665355, 4, 148, 928),
 (4669457, 4, 732, 936),
 (4673552, 4, 264, 948),
 (4677639, 4, 478, 949),
 (4685833, 4, 123, 919),
 (4685834, 4, 475, 961),
 (4722705, 4, 732, 936),
 (4726802, 4, 6066, 955),
 (4730888, 1, 2233, 899),
 (4730888, 1, 7998, 899),
 (

In [31]:
from collections import defaultdict, Counter

ANCHOR_TOL = 2

db_map = defaultdict(list)
for h, song_id, offset, anchor_freq in results:
    db_map[int(h)].append((int(song_id), int(offset), int(anchor_freq)))

votes = defaultdict(list)

for h, q_offset, q_anchor_freq in hashes:
    h = int(h)
    q_offset = int(q_offset)
    q_anchor_freq = int(q_anchor_freq)

    candidates = db_map.get(h, [])
    for song_id, db_offset, db_anchor_freq in candidates:
        if abs(db_anchor_freq - q_anchor_freq) > ANCHOR_TOL:
            continue
        delta = db_offset - q_offset
        votes[song_id].append(delta)

best_song = None
best_score = 0

for song_id, deltas in votes.items():
    counter = Counter(deltas)
    _, count = counter.most_common(1)[0]

    if count > best_score:
        best_score = count
        best_song = song_id

best_title = None
if best_song is not None:
    conn = psycopg2.connect(os.getenv("DATABASE_URL"))
    cur = conn.cursor()
    cur.execute("SELECT title FROM songs WHERE id = %s", (best_song,))
    row = cur.fetchone()
    best_title = row[0] if row else None
    cur.close()
    conn.close()

{"song_id": best_song, "title": best_title, "score": best_score, "anchor_tol": ANCHOR_TOL}

{'song_id': 4, 'title': 'Kaun Tujhe', 'score': 69092, 'anchor_tol': 2}

In [ ]:
import os
import tempfile
from collections import defaultdict, Counter

import librosa
import numpy as np
import soundfile as sf
import psycopg2

from app.utils import generate_hashes_for_audio

def match_chunk_variant(audio_path, start_sec=30.0, chunk_sec=12.0, pitch_steps=0.0, gain=1.0, anchor_tol=2, min_score=10):
    y, sr = librosa.load(audio_path, sr=22050)

    start = int(start_sec * sr)
    end = min(len(y), start + int(chunk_sec * sr))
    if start >= len(y) or start >= end:
        return {
            "variant": {"start_sec": start_sec, "chunk_sec": chunk_sec, "pitch_steps": pitch_steps, "gain": gain},
            "error": "Invalid chunk range",
        }

    chunk = y[start:end]

    if pitch_steps != 0:
        chunk = librosa.effects.pitch_shift(chunk, sr=sr, n_steps=pitch_steps)

    if gain != 1.0:
        chunk = np.clip(chunk * gain, -1.0, 1.0)

    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        temp_path = tmp.name
    sf.write(temp_path, chunk, sr)

    try:
        q_hashes = generate_hashes_for_audio(temp_path)
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)

    if q_hashes.size == 0:
        return {
            "variant": {"start_sec": start_sec, "chunk_sec": chunk_sec, "pitch_steps": pitch_steps, "gain": gain},
            "hash_count": 0,
            "match": None,
            "score": 0,
        }

    query_hashes = q_hashes[:, 0].astype(int).tolist()

    conn = psycopg2.connect(os.getenv("DATABASE_URL"))
    cur = conn.cursor()
    cur.execute(
        """
        SELECT hash, song_id, time_offset, anchor_freq
        FROM fingerprints
        WHERE hash = ANY(%s)
        """,
        (query_hashes,),
    )
    rows = cur.fetchall()
    cur.close()
    conn.close()

    db_map = defaultdict(list)
    for h, song_id, offset, anchor_freq in rows:
        db_map[int(h)].append((int(song_id), int(offset), int(anchor_freq)))

    votes = defaultdict(list)
    for h, q_offset, q_anchor_freq in q_hashes:
        h = int(h)
        q_offset = int(q_offset)
        q_anchor_freq = int(q_anchor_freq)

        candidates = db_map.get(h, [])
        for song_id, db_offset, db_anchor_freq in candidates:
            if abs(db_anchor_freq - q_anchor_freq) > anchor_tol:
                continue
            votes[song_id].append(db_offset - q_offset)

    best_song = None
    best_score = 0
    for song_id, deltas in votes.items():
        count = Counter(deltas).most_common(1)[0][1]
        if count > best_score:
            best_score = count
            best_song = song_id

    if best_score < min_score:
        best_song = None

    best_title = None
    if best_song is not None:
        conn = psycopg2.connect(os.getenv("DATABASE_URL"))
        cur = conn.cursor()
        cur.execute("SELECT title FROM songs WHERE id = %s", (best_song,))
        row = cur.fetchone()
        best_title = row[0] if row else None
        cur.close()
        conn.close()

    return {
        "variant": {
            "start_sec": start_sec,
            "chunk_sec": chunk_sec,
            "pitch_steps": pitch_steps,
            "gain": gain,
            "anchor_tol": anchor_tol,
            "min_score": min_score,
        },
        "hash_count": int(len(query_hashes)),
        "song_id": best_song,
        "title": best_title,
        "score": int(best_score),
    }

audio_path = "data/misirlou.mp3"
tests = [
    {"start_sec": 30, "chunk_sec": 12, "pitch_steps": 0, "gain": 1.0, "anchor_tol": 2, "min_score": 10},
    {"start_sec": 30, "chunk_sec": 12, "pitch_steps": 2, "gain": 1.0, "anchor_tol": 2, "min_score": 10},
    {"start_sec": 30, "chunk_sec": 12, "pitch_steps": -2, "gain": 1.0, "anchor_tol": 2, "min_score": 10},
    {"start_sec": 30, "chunk_sec": 12, "pitch_steps": 0, "gain": 0.6, "anchor_tol": 2, "min_score": 10},
    {"start_sec": 30, "chunk_sec": 12, "pitch_steps": 0, "gain": 1.4, "anchor_tol": 2, "min_score": 10},
]

outputs = [match_chunk_variant(audio_path, **cfg) for cfg in tests]
outputs

[{'variant': {'start_sec': 30,
   'chunk_sec': 12,
   'pitch_steps': 0,
   'gain': 1.0,
   'anchor_tol': 2,
   'min_score': 10},
  'hash_count': 5566,
  'song_id': 2,
  'title': 'Misirlou',
  'score': 4712},
 {'variant': {'start_sec': 30,
   'chunk_sec': 12,
   'pitch_steps': 2,
   'gain': 1.0,
   'anchor_tol': 2,
   'min_score': 10},
  'hash_count': 6026,
  'song_id': None,
  'title': None,
  'score': 3},
 {'variant': {'start_sec': 30,
   'chunk_sec': 12,
   'pitch_steps': -2,
   'gain': 1.0,
   'anchor_tol': 2,
   'min_score': 10},
  'hash_count': 5293,
  'song_id': None,
  'title': None,
  'score': 3},
 {'variant': {'start_sec': 30,
   'chunk_sec': 12,
   'pitch_steps': 0,
   'gain': 0.6,
   'anchor_tol': 2,
   'min_score': 10},
  'hash_count': 5566,
  'song_id': 2,
  'title': 'Misirlou',
  'score': 4712},
 {'variant': {'start_sec': 30,
   'chunk_sec': 12,
   'pitch_steps': 0,
   'gain': 1.4,
   'anchor_tol': 2,
   'min_score': 10},
  'hash_count': 5576,
  'song_id': 2,
  'title': '